In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression

In [3]:
df = pd.read_csv("../data/raw/career_dataset.csv")

In [4]:
df.head()


,skills,education,interests,experience,career
0,numpy git statistics linux pandas machine lear...,BTech CSE,machine learning,3,Data Scientist
1,pandas numpy data visualization aws,BSc Statistics,machine learning,7,Data Scientist
2,data visualization numpy statistics linux sql ...,MSc Data Science,research,1,Data Scientist
3,python data visualization numpy statistics pro...,BTech CSE,problem solving,5,Data Scientist
4,python numpy statistics communication sql pand...,MCA,innovation,5,Data Scientist


In [5]:
X = df.drop("career", axis=1)
y = df["career"]

In [6]:
df.head()

,skills,education,interests,experience,career
0,numpy git statistics linux pandas machine lear...,BTech CSE,machine learning,3,Data Scientist
1,pandas numpy data visualization aws,BSc Statistics,machine learning,7,Data Scientist
2,data visualization numpy statistics linux sql ...,MSc Data Science,research,1,Data Scientist
3,python data visualization numpy statistics pro...,BTech CSE,problem solving,5,Data Scientist
4,python numpy statistics communication sql pand...,MCA,innovation,5,Data Scientist


In [8]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [9]:
dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

{'AI Engineer': np.int64(0),
 'Backend Developer': np.int64(1),
 'Business Analyst': np.int64(2),
 'Cloud Engineer': np.int64(3),
 'Cybersecurity Analyst': np.int64(4),
 'Data Scientist': np.int64(5),
 'DevOps Engineer': np.int64(6),
 'Frontend Developer': np.int64(7),
 'Full Stack Developer': np.int64(8),
 'Machine Learning Engineer': np.int64(9)}

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("skills_tfidf", TfidfVectorizer(), "skills"),
        ("education_ohe", OneHotEncoder(handle_unknown="ignore"), ["education"]),
        ("interests_tfidf", TfidfVectorizer(), "interests"),
        ("experience_num", "passthrough", ["experience"])
    ]
)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [12]:
model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])


In [13]:
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('skills_tfidf', ...), ('education_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [14]:
y_pred = model_pipeline.predict(X_test)

In [15]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 1.0


In [16]:
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

                           precision    recall  f1-score   support

              AI Engineer       1.00      1.00      1.00        20
        Backend Developer       1.00      1.00      1.00        20
         Business Analyst       1.00      1.00      1.00        20
           Cloud Engineer       1.00      1.00      1.00        20
    Cybersecurity Analyst       1.00      1.00      1.00        20
           Data Scientist       1.00      1.00      1.00        20
          DevOps Engineer       1.00      1.00      1.00        20
       Frontend Developer       1.00      1.00      1.00        20
     Full Stack Developer       1.00      1.00      1.00        20
Machine Learning Engineer       1.00      1.00      1.00        20

                 accuracy                           1.00       200
                macro avg       1.00      1.00      1.00       200
             weighted avg       1.00      1.00      1.00       200



In [17]:
cm = confusion_matrix(y_test, y_pred)
cm

array([[20,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0, 20,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0, 20,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0, 20,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0, 20,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0, 20,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0, 20,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0, 20,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0, 20,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 20]])

In [18]:
sample_user = pd.DataFrame([{
    "skills": "python sql machine learning tensorflow docker",
    "education": "BTech CSE",
    "interests": "ai",
    "experience": 2
}])

In [19]:
prediction = model_pipeline.predict(sample_user)
predicted_career = label_encoder.inverse_transform(prediction)

print(predicted_career)

['Machine Learning Engineer']


In [20]:
sample_user = pd.DataFrame([{
    "skills": "python sql machine learning tensorflow docker",
    "education": "BTech CSE",
    "interests": "ai",
    "experience": 2
}])

In [21]:
probabilities = model_pipeline.predict_proba(sample_user)

In [22]:
probabilities

array([[0.17164349, 0.0447325 , 0.01224666, 0.0271012 , 0.01845668,
        0.18464003, 0.0298962 , 0.00625255, 0.01910794, 0.48592274]])

In [23]:

label_encoder.classes_

array(['AI Engineer', 'Backend Developer', 'Business Analyst',
       'Cloud Engineer', 'Cybersecurity Analyst', 'Data Scientist',
       'DevOps Engineer', 'Frontend Developer', 'Full Stack Developer',
       'Machine Learning Engineer'], dtype=object)

In [25]:
array = np.array([
 'AI Engineer',
 'Backend Developer',
 'Business Analyst',
 ...
])

In [26]:
career_probabilities = list(zip(label_encoder.classes_, probabilities[0]))
career_probabilities

[('AI Engineer', np.float64(0.17164349408879095)),
 ('Backend Developer', np.float64(0.044732497746795254)),
 ('Business Analyst', np.float64(0.012246664440092375)),
 ('Cloud Engineer', np.float64(0.02710119536302547)),
 ('Cybersecurity Analyst', np.float64(0.018456683295413445)),
 ('Data Scientist', np.float64(0.1846400338675542)),
 ('DevOps Engineer', np.float64(0.02989620440169209)),
 ('Frontend Developer', np.float64(0.006252547262579956)),
 ('Full Stack Developer', np.float64(0.01910793516075038)),
 ('Machine Learning Engineer', np.float64(0.48592274437330596))]

In [31]:
career_probabilities = sorted(
    career_probabilities,
    key=lambda x: x[1],
    reverse=True
)

In [32]:
career_probabilities

[('Machine Learning Engineer', np.float64(0.48592274437330596)),
 ('Data Scientist', np.float64(0.1846400338675542)),
 ('AI Engineer', np.float64(0.17164349408879095)),
 ('Backend Developer', np.float64(0.044732497746795254)),
 ('DevOps Engineer', np.float64(0.02989620440169209)),
 ('Cloud Engineer', np.float64(0.02710119536302547)),
 ('Full Stack Developer', np.float64(0.01910793516075038)),
 ('Cybersecurity Analyst', np.float64(0.018456683295413445)),
 ('Business Analyst', np.float64(0.012246664440092375)),
 ('Frontend Developer', np.float64(0.006252547262579956))]

In [33]:
top_3 = career_probabilities[:3]

for career, score in top_3:
    print(f"{career}: {score*100:.2f}%")

Machine Learning Engineer: 48.59%
Data Scientist: 18.46%
AI Engineer: 17.16%


In [35]:
top_career = top_3[0][0]
top_career

'Machine Learning Engineer'

In [39]:
career_profiles = {
    "Data Scientist": {
        "skills": [
            "python", "sql", "pandas", "numpy",
            "machine learning", "statistics", "data visualization"
        ],
        "education": [
            "BTech CSE", "MCA", "MSc Data Science", "BSc Statistics"
        ],
        "interests": [
            "ai", "data analytics", "research", "machine learning"
        ]
    },

    "Frontend Developer": {
        "skills": [
            "html", "css", "javascript", "react",
            "bootstrap", "ui design"
        ],
        "education": [
            "BCA", "BTech IT", "MCA"
        ],
        "interests": [
            "web design", "frontend", "ui ux", "creative coding"
        ]
    },

    "Backend Developer": {
        "skills": [
            "python", "java", "sql", "django",
            "flask", "api development"
        ],
        "education": [
            "BTech CSE", "MCA", "BCA"
        ],
        "interests": [
            "backend systems", "databases", "server architecture"
        ]
    },

    "Machine Learning Engineer": {
        "skills": [
            "python", "tensorflow", "pytorch",
            "machine learning", "deep learning", "numpy"
        ],
        "education": [
            "BTech CSE", "MTech AI", "MCA"
        ],
        "interests": [
            "ai", "deep learning", "automation"
        ]
    },

    "Cloud Engineer": {
        "skills": [
            "aws", "docker", "linux", "kubernetes",
            "terraform", "networking"
        ],
        "education": [
            "BTech CSE", "BTech IT", "MCA"
        ],
        "interests": [
            "cloud computing", "infrastructure", "devops"
        ]
    },

    "Cybersecurity Analyst": {
        "skills": [
            "network security", "ethical hacking", "linux",
            "penetration testing", "python"
        ],
        "education": [
            "BTech CSE", "BSc Cybersecurity", "MCA"
        ],
        "interests": [
            "security", "ethical hacking", "cyber defense"
        ]
    },

    "Business Analyst": {
        "skills": [
            "excel", "sql", "power bi",
            "data analysis", "communication"
        ],
        "education": [
            "MBA", "BBA", "BCom"
        ],
        "interests": [
            "business strategy", "analytics", "reporting"
        ]
    },

    "DevOps Engineer": {
        "skills": [
            "docker", "jenkins", "linux", "aws",
            "kubernetes", "ci cd"
        ],
        "education": [
            "BTech CSE", "MCA"
        ],
        "interests": [
            "automation", "deployment", "infrastructure"
        ]
    },

    "AI Engineer": {
        "skills": [
            "python", "machine learning", "deep learning",
            "nlp", "tensorflow", "llms"
        ],
        "education": [
            "BTech CSE", "MTech AI", "MCA"
        ],
        "interests": [
            "artificial intelligence", "nlp", "automation"
        ]
    },

    "Full Stack Developer": {
        "skills": [
            "html", "css", "javascript", "react",
            "node js", "mongodb", "express"
        ],
        "education": [
            "BTech CSE", "BCA", "MCA"
        ],
        "interests": [
            "web development", "frontend", "backend"
        ]
    }
}

In [40]:
top_career = top_3[0][0]
ideal_skills = set(career_profiles[top_career]["skills"])
ideal_skills

{'deep learning',
 'machine learning',
 'numpy',
 'python',
 'pytorch',
 'tensorflow'}

In [41]:
ideal_skills = set(career_profiles[top_career]["skills"])
ideal_skills

{'deep learning',
 'machine learning',
 'numpy',
 'python',
 'pytorch',
 'tensorflow'}

In [42]:
user_skills = set(sample_user.iloc[0]["skills"].split())
user_skills

{'docker', 'learning', 'machine', 'python', 'sql', 'tensorflow'}

In [43]:
missing_skills = ideal_skills - user_skills
missing_skills

{'deep learning', 'machine learning', 'numpy', 'pytorch'}

In [44]:
print("Recommended Career:", top_career)
print("\nMissing Skills:")

for skill in missing_skills:
    print("-", skill)

Recommended Career: Machine Learning Engineer

Missing Skills:
- deep learning
- numpy
- pytorch
- machine learning


In [45]:
user_skills = set(skill.strip() for skill in sample_user.iloc[0]["skills"].split())

In [46]:
sample_user = pd.DataFrame([{
    "skills": "python, sql, machine learning, tensorflow, docker",
    "education": "BTech CSE",
    "interests": "ai",
    "experience": 2
}])

In [47]:
user_skills = set(
    skill.strip().lower()
    for skill in sample_user.iloc[0]["skills"].split(",")
)

In [48]:
ideal_skills = set(
    skill.lower()
    for skill in career_profiles[top_career]["skills"]
)

In [49]:
missing_skills = ideal_skills - user_skills

In [50]:
missing_skills

{'deep learning', 'numpy', 'pytorch'}

In [51]:
print("Recommended Career:", top_career)
print("\nMissing Skills:")

for skill in missing_skills:
    print("-", skill)

Recommended Career: Machine Learning Engineer

Missing Skills:
- deep learning
- numpy
- pytorch


In [52]:
def recommend_career(skills, education, interests, experience):
    
    user_df = pd.DataFrame([{
        "skills": skills,
        "education": education,
        "interests": interests,
        "experience": experience
    }])

    probabilities = model_pipeline.predict_proba(user_df)

    career_probabilities = list(
        zip(label_encoder.classes_, probabilities[0])
    )

    career_probabilities = sorted(
        career_probabilities,
        key=lambda x: x[1],
        reverse=True
    )

    top_3 = career_probabilities[:3]

    top_career = top_3[0][0]

    ideal_skills = set(
        skill.lower()
        for skill in career_profiles[top_career]["skills"]
    )

    user_skills = set(
        skill.strip().lower()
        for skill in skills.split(",")
    )

    missing_skills = list(ideal_skills - user_skills)

    formatted_top_3 = [
        (career, round(score * 100, 2))
        for career, score in top_3
    ]

    return {
        "recommended_career": top_career,
        "top_3_recommendations": formatted_top_3,
        "missing_skills": missing_skills
    }

In [56]:
result = recommend_career(
    skills="python, sql, machine learning, tensorflow, docker",
    education="BTech CSE",
    interests="ai",
    experience=2
)

In [57]:
result

{'recommended_career': 'Machine Learning Engineer',
 'top_3_recommendations': [('Machine Learning Engineer', np.float64(48.59)),
  ('Data Scientist', np.float64(18.46)),
  ('AI Engineer', np.float64(17.16))],
 'missing_skills': ['deep learning', 'numpy', 'pytorch']}

In [58]:
print("Recommended Career:", result["recommended_career"])

print("\nTop 3 Recommendations:")
for career, score in result["top_3_recommendations"]:
    print(f"{career}: {score}%")

print("\nMissing Skills:")
for skill in result["missing_skills"]:
    print("-", skill)

Recommended Career: Machine Learning Engineer

Top 3 Recommendations:
Machine Learning Engineer: 48.59%
Data Scientist: 18.46%
AI Engineer: 17.16%

Missing Skills:
- deep learning
- numpy
- pytorch


In [59]:
import joblib

In [60]:
joblib.dump(model_pipeline, "../models/career_model.pkl")

['../models/career_model.pkl']

In [61]:
joblib.dump(label_encoder, "../models/label_encoder.pkl")

['../models/label_encoder.pkl']

In [62]:
import os
os.listdir("../models")

['scaler.pkl',
 'label_encoder.pkl',
 'encoder.pkl',
 'career_model.pkl',
 'trained_model.pkl']

In [63]:
loaded_model = joblib.load("../models/career_model.pkl")
loaded_encoder = joblib.load("../models/label_encoder.pkl")

In [64]:
test_result = loaded_model.predict(sample_user)
loaded_encoder.inverse_transform(test_result)

array(['Machine Learning Engineer'], dtype=object)